In [1]:
import pandas as pd
import numpy as np
from typing import Tuple, List
from sklearn.model_selection import train_test_split, validation_curve, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

In [2]:
data = pd.read_csv("train.csv")
print("data.shape = {} rows, {} cols".format(*data.shape))
data.head(n=2)

data.shape = 60000 rows, 202 cols


,ID_code,target,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,...,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199
0,train_0,0,8.9255,-6.7863,11.9081,5.093,11.4607,-9.2834,5.1187,18.6266,...,4.4354,3.9642,3.1364,1.6910,18.5227,-2.3978,7.8784,8.5635,12.7803,-1.0914
1,train_1,0,11.5006,-4.1473,13.8588,5.389,12.3622,7.0433,5.6208,16.5338,...,7.6421,7.7214,2.5837,10.9516,15.4305,2.0339,8.1267,8.7889,18.3560,1.9518


In [3]:
x_train, x_valid = train_test_split(
    data.drop(["ID_code", "target"], axis=1), train_size=0.75, shuffle=True, random_state=1,
)
y_train, y_valid = train_test_split(
    data["target"], train_size=0.75, shuffle=True, random_state=1,
)
print("x_train.shape = {} rows, {} cols".format(*x_train.shape))
print("x_valid.shape = {} rows, {} cols".format(*x_valid.shape))

x_train.shape = 45000 rows, 200 cols
x_valid.shape = 15000 rows, 200 cols


In [4]:
def fit_evaluate_model(estimator, x_train, y_train, x_valid, y_valid):
    """
    Функция для обучения и оценки качества модели.

    Parameters
    ----------
    estimator: callable
        Объект для обучения и применения модели.

    x_train: pandas.DataFrame
        Матрица признаков для обучения модели.

    y_train: pandas.Series
        Вектор целевой переменной для обучения модели.

    x_valid: pandas.DataFrame
        Матрица признаков для валидации модели.

    y_valid: pandas.Series
        Вектор целевой переменной для валидации модели.

    Returns
    -------
    y_train_pred: np.array
        Вектор прогнозов для обучающей выборки

    y_valid_pred: np.array
        Вектор прогнозов для валидационной выборки

    """
    estimator.fit(x_train, y_train)
    y_train_pred = estimator.predict_proba(x_train)[:, 1]
    y_valid_pred = estimator.predict_proba(x_valid)[:, 1]

    train_score = roc_auc_score(y_train, y_train_pred)
    valid_score = roc_auc_score(y_valid, y_valid_pred)
    print(f"Model Score: train = {round(train_score, 4)}, valid = {round(valid_score, 4)}")

    return y_train_pred, y_valid_pred

### Линейная модель

In [5]:
pipeline = Pipeline(
    steps=[
        ("scaling", StandardScaler()),
        ("model", LogisticRegression(random_state=27, C=1e-5))
    ]
)
y_train_pred, y_valid_pred = fit_evaluate_model(
    pipeline, x_train, y_train, x_valid, y_valid
)

Model Score: train = 0.8633, valid = 0.8476


### Bagging над линейными моделями

In [6]:
bagging = BaggingClassifier(
    estimator=pipeline, random_state=27, n_jobs=2
)
y_train_pred, y_valid_pred = fit_evaluate_model(
    bagging, x_train, y_train, x_valid, y_valid
)

Model Score: train = 0.8631, valid = 0.8474


________________________________________________________________________

### Решающее дерево без ограничения по глубине

In [7]:
tree = DecisionTreeClassifier(
    random_state=27
)
y_train_pred, y_valid_pred = fit_evaluate_model(
    tree, x_train, y_train, x_valid, y_valid
)

Model Score: train = 1.0, valid = 0.5567


### Bagging над решающими деревьями неограниченной глубины

In [8]:
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=27), random_state=27, n_jobs=2
)
y_train_pred, y_valid_pred = fit_evaluate_model(
    bagging, x_train, y_train, x_valid, y_valid
)

Model Score: train = 0.9998, valid = 0.6881


________________________________________________________________________

### Решающее дерево с ограничением по глубине

In [9]:
tree = DecisionTreeClassifier(
    max_depth=5, random_state=27
)
y_train_pred, y_valid_pred = fit_evaluate_model(
    tree, x_train, y_train, x_valid, y_valid
)

Model Score: train = 0.6452, valid = 0.606


### Bagging над решающими деревьями ограниченными по глубине

In [10]:
%%time
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=5, random_state=27), random_state=27, n_jobs=2
)
y_train_pred, y_valid_pred = fit_evaluate_model(
    bagging, x_train, y_train, x_valid, y_valid
)

Model Score: train = 0.7215, valid = 0.6764
CPU times: user 49.4 ms, sys: 56.4 ms, total: 106 ms
Wall time: 12.7 s


________________________________________________________________________

### Случайный лес

In [11]:
%%time
forest = RandomForestClassifier(
    random_state=27
)
y_train_pred, y_valid_pred = fit_evaluate_model(
    forest, x_train, y_train, x_valid, y_valid
)

Model Score: train = 1.0, valid = 0.7978
CPU times: user 2min 3s, sys: 345 ms, total: 2min 3s
Wall time: 2min 3s
